In [1]:
#На кодах Гоппы и с матрицей перестановки P
#Может шифровать только сообщения малой длины.
#Поэтому её используют для подписи. Если для шифрования сообщения
#То в связке с KEM+DEM (aes внутри для шифрования )

In [2]:
def random_permutation_matrix(n, F):
    perm = list(range(n))
    shuffle(perm)
    P = matrix(F, n, n, lambda i, j: 1 if j == perm[i] else 0)
    return P, perm
    
F2 = GF(2)

# === Параметры Goppa-кода ===
m = 4                    # GF(2^m) = GF(16)
t = 2                    # исправляет t ошибок
Fq = GF(2^m, 'a')        # явно указываем имя генератора

# Многочлен для построения поля (лучше фиксированный)
Fq = GF(2^m, name='a', modulus=x^4 + x + 1)  # стандартный примитивный полином

R.<x> = PolynomialRing(Fq)

# Ищем неприводимый g(x) = x^2 + a*x + b в Fq[x]
g = None
for a_coeff in Fq:
    for b_coeff in Fq:
        cand = x^2 + a_coeff*x + b_coeff
        if cand.is_irreducible():
            g = cand
            break
    if g:
        break

if g is None:
    raise RuntimeError("Не найден неприводимый многочлен степени 2")

print("Goppa-полином g(x) =", g)

# Локаторы: все α ∈ Fq, такие что g(α) ≠ 0
L = [alpha for alpha in Fq if g(alpha) != 0]

# Создаём код
C = codes.GoppaCode(g, L)
k = C.dimension()
n = C.length()
d = C.minimum_distance()
t = (d - 1) // 2
r = n - k
print(f"Код Гоппы: [n={n}, k={k}, r={r}], t={t}")
print("Мин. расстояние ≥ 2t+1 =", 2*t+1)

# Получаем H над GF(2)!
H = C.parity_check_matrix()
H = H.change_ring(F2)   # на всякий случай
assert H.base_ring() == F2
print("H ∈ GF(2)^({}×{})".format(H.nrows(), H.ncols()))

# Проверка ранга
assert H.rank() == r, "Ранг H не совпадает с r"


G = C.generator_matrix()   # k x n, над GF(2)
print(f"Код Гоппы: [{n}, {k}, d≥{2*t+1}] → t = {t}")
print("Порождающая матрица G (первые 3 строки):")
print(G[:3])  # не печатаем всю, если большая

# --- 2. Генерируем S (n-k x n-k, невырожденную) ---
S = random_matrix(F2, n-k, n-k)
while not S.is_invertible():
    S = random_matrix(F2, n-k, n-k)
print(f"\nМатрица S (n-k x n-k): {S.dimensions()}")
print("S.is_invertible() =", S.is_invertible())

# --- 3. Генерируем P (n x n) ---
P, perm = random_permutation_matrix(n, F2)
print("Матрица P:", P.dimensions())

Goppa-полином g(x) = x^2 + a*x + a
Код Гоппы: [n=16, k=8, r=8], t=2
Мин. расстояние ≥ 2t+1 = 5
H ∈ GF(2)^(8×16)
Код Гоппы: [16, 8, d≥5] → t = 2
Порождающая матрица G (первые 3 строки):
[1 0 0 0 0 0 0 0 0 1 0 1 1 0 0 1]
[0 1 0 0 0 0 0 0 1 0 1 0 1 0 1 1]
[0 0 1 0 0 0 0 0 1 1 1 1 0 0 0 1]

Матрица S (n-k x n-k): (8, 8)
S.is_invertible() = True
Матрица P: (16, 16)


In [3]:
H

[1 1 0 0 1 0 1 1 0 0 0 1 1 1 1 1]
[0 0 1 0 0 1 1 1 0 0 0 1 1 1 1 0]
[0 0 0 1 0 0 1 0 1 1 1 1 0 0 0 0]
[1 1 0 0 0 0 0 0 1 0 1 0 1 1 0 0]
[0 1 0 0 1 0 0 0 1 0 0 0 1 0 0 1]
[0 0 0 1 1 0 1 1 0 1 1 0 1 1 0 0]
[0 0 0 1 0 1 0 1 0 1 0 1 0 1 0 0]
[0 0 1 0 0 1 0 1 1 1 0 1 0 0 1 0]

In [4]:
n-k,n

(8, 16)

In [5]:
S

[1 0 1 1 0 0 1 0]
[0 1 1 1 0 0 0 0]
[0 1 1 1 0 0 1 1]
[1 0 1 0 1 0 0 1]
[0 0 0 0 1 0 1 0]
[1 0 1 0 0 1 1 0]
[1 0 0 1 0 1 0 0]
[0 1 1 0 0 1 1 0]

In [6]:
n-k

8

In [7]:
Hpub = S * H * P #ТУТ МАТРИЦА ПЕРЕСТАНОВКИ P, БЕЗ НЕЁ ВЗЛАМЫВАЕТСЯ

In [8]:
Hpub

[0 1 1 1 0 0 0 1 1 0 1 0 0 0 0 0]
[1 0 0 0 0 1 1 0 1 1 1 0 1 0 0 1]
[0 0 1 0 1 0 1 0 1 1 0 0 1 0 0 1]
[1 1 1 0 1 1 0 0 1 0 0 0 1 0 1 0]
[0 1 1 1 1 1 1 1 1 1 0 0 0 1 0 1]
[0 1 1 0 1 1 1 1 1 1 1 1 1 0 0 1]
[0 1 1 0 1 1 0 1 0 1 1 0 0 1 0 0]
[1 1 1 1 1 1 0 0 0 1 1 1 0 0 0 1]

In [10]:
pub = (Hpub, t)

In [11]:
S_inv = S.inverse()
P_inv = P.inverse()

In [12]:
priv = (S_inv, H, P_inv)

In [13]:
#Шифрование

In [17]:
from itertools import combinations

In [18]:
n = Hpub.ncols()
t = 2  # как у тебя

# Генерируем все возможные пары позиций
all_pairs = list(combinations(range(n), t))
idx = ord('h')  # 104
if idx >= len(all_pairs):
    idx = len(all_pairs) - 1  # обрезаем

pos1, pos2 = all_pairs[idx]
m = vector(F2, [1 if i in (pos1, pos2) else 0 for i in range(n)])

print(f"Сообщение m (вес = {m.hamming_weight()}):")
print(m.list())

Сообщение m (вес = 2):
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1]


In [19]:
# --- Шаг 2: Боб вычисляет шифротекст c = m * Hpub^T ---
c = m * Hpub.transpose()  # ← важно: транспонирование!
c = vector(F2, c.list())  # шифротекст — вектор длины r

print(f"Шифротекст c (длина {len(c)}):")
print(c.list())

Шифротекст c (длина 8):
[0, 0, 0, 0, 0, 0, 1, 0]


In [20]:
#Расшифрование

In [21]:
# 1. s = c * S_inv^T
S_inv_T = S_inv.transpose()
s = c * S_inv_T
s = vector(F2, s.list())

# 2. Найти hat_m_col (столбец), такой что H * hat_m_col = s.column()
try:
    D = C.decoder('Syndrome')
    hat_m_col = D.decode_to_error_vector(s)  # возвращает столбец? нет — вектор
except:
    from itertools import combinations
    for pos in combinations(range(n), t):
        cand = vector(F2, [1 if i in pos else 0 for i in range(n)])
        if H * cand.column() == s.column():
            hat_m_col = cand
            break
    else:
        raise ValueError("Ошибка")

# 3. m^T = P^{-1} * hat_m_col = P^T * hat_m_col
#    → m = hat_m_col^T * P
m_dec = hat_m_col * P   # ← строковый вектор × матрица
m_dec = vector(F2, m_dec.list())

In [22]:
m

(0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1)

In [23]:
m_dec == m

True